### Data Quality Assessment & Cleaning

### Theme

> **Analyze First → Clean Second**

---

This is one of the **most important sessions** in the entire Machine Learning lifecycle. One mistake many beginner ML engineers make is immediately writing:

```python
df.dropna()
```

or

```python
df.fillna()
```

without first understanding **what is wrong with the data**.

A professional ML engineer spends a significant amount of time answering:

> **"What problems exist in my data?"**

before asking:

> **"How do I fix them?"**

This session teaches exactly that mindset.

---

### Learning objectives

By the end of this lesson students should be able to:

* Explain why data quality matters.
* Identify common dataset problems.
* Measure data quality.
* Decide whether data should be removed or repaired.
* Apply appropriate cleaning techniques.
* Understand why every cleaning decision affects model performance.

---

### Business scenario

Imagine your company collected incident reports from multiple oil fields.

Reports came from:

* Mobile applications
* Excel sheets
* Email submissions
* Paper forms entered manually

Because many people entered the reports, the dataset now contains many inconsistencies.

Your job is **NOT to build the model yet.** Your first job is to inspect the dataset and prepare it for machine learning.

---

### Step 1 - Create a "Dirty" Dataset

Replace your existing CSV with the following intentionally messy data.

```csv
incident_id,report_text,location,reported_by
1,"Pressure leak detected on Pipeline 7.",Pipeline 7,John
2,"Worker slipped near drilling platform.",Platform A,Aisha
3,"Gas detector triggered alarm.",Plant B,Michael
4,"Small oil spill observed near storage tank.",Tank Farm,David
5,"Emergency shutdown initiated after abnormal vibration.",Refinery,Grace
6,,Pipeline 8,John
7,"   Pressure leak detected on Pipeline 7.   ",Pipeline 7,John
8,"gas detector triggered alarm.",Plant B,Michael
9,"Gas detector triggred alarm.",Plant B,Michael
10,"Worker slipped near drilling platform.",Platform A,Aisha
11,"",Tank Farm,David
12,"PRESSURE LEAK DETECTED ON PIPELINE 7.",Pipeline 7,John
13,"Pressure leak detected on Pipeline 7.",Pipeline 7,John
14,"Worker slipped near drilling platform.",Platform A,
15,"Emergency shutdown initiated after abnormal vibration.",Refinery,Grace
```

Note:

> **This dataset is intentionally "dirty."**

---

### Step 2 - Do NOT clean anything

Instead:

> **"What looks suspicious?"**

You should inspect before modifying.

---

### Step 3 - Use `DataExplorer`

Extend your `DataExplorer` class.

```python
import pandas as pd


class DataExplorer:

    def dataset_shape(self, df):
        print("Dataset Shape:")
        print(df.shape)

    def show_columns(self, df):
        print("\nColumns:")
        print(df.columns)

    def data_info(self, df):
        print("\nInformation:")
        print(df.info())

    def preview(self, df):
        print("\nFirst Five Rows:")
        print(df.head())

    def random_samples(self, df):
        print("\nRandom Samples:")
        print(df.sample(5))
```

---

### Question?

Without writing any cleaning code, identify problems.

---

### Problem 1 - Missing values

Learn:

```python
df.isnull().sum()
```

Output might be

```text
incident_id      0
report_text      2
location         0
reported_by      1
```

Ask:

Which columns contain missing values?

---

### Why is this dangerous?

If the model receives

```text
NaN
```

instead of

```text
Pressure leak detected...
```

it cannot learn meaningful patterns.

---

### How to fix `missing values`

Option 1:

Remove rows

```python
df = df.dropna()
```

Discussion:

When is removing rows acceptable?

* Small number of missing records.
* Missing data is not critical.

---

Option 2:

Replace missing values

```python
df["reported_by"] = df["reported_by"].fillna("Unknown")
```

---

Option 3:

Replace missing report text

```python
df["report_text"] = df["report_text"].fillna("No Report")
```

Note:

Sometimes replacing text is useful for preserving records, but it may not always be appropriate for model training. The decision depends on the problem.

---

### Problem 2 - Empty strings

Missing values are NOT the same as

```text
""
```
Learn:

```python
df[df["report_text"] == ""]
```

Also teach

```python
df["report_text"].str.strip() == ""
```

---

### Fix

```python
df["report_text"] = df["report_text"].replace("", pd.NA)
```

Then

```python
df = df.dropna(subset=["report_text"])
```

---

### Problem 3 - Duplicate records

Learn:

```python
df.duplicated()
```

Then

```python
df[df.duplicated()]
```

Question?

Why are duplicates dangerous?

Because: One incident now counts twice. The model becomes biased.

---

### Fix

```python
df = df.drop_duplicates()
```

---

### Problem 4 - Extra whitespace

Example

```text
"   Pressure leak detected..."
```

looks identical

but

Python disagrees.

Learn:

```python
df["report_text"]
```

You may not notice.

But when used this:

```python
repr(df.loc[6, "report_text"])
```

Now, you'll see spaces.

---

### Fix

```python
df["report_text"] = df["report_text"].str.strip()
```

---

### Problem 5 - Inconsistent capitalization

Examples

```text
Gas detector triggered alarm.

gas detector triggered alarm.

GAS DETECTOR TRIGGERED ALARM.
```

To us this gives same meaning. To machine learning it's three different sentences.

---

Learn:

```python
df["report_text"]
```

You should identify inconsistency.

---

### Fix

```python
df["report_text"] = df["report_text"].str.lower()
```

Now, everything becomes lower cased.

```text
gas detector triggered alarm.
```

---

### Problem 6 - Typographical errors

Example

```text
triggred
```

instead of

```text
triggered
```

Ask

Can Python detect spelling mistakes?

No.

---

Possible solutions:

- Dictionary

- Manual corrections

- Spell checker

- Language model

Today We'll use a dictionary.

```python
corrections = {
    "triggred": "triggered"
}
```

---

`Fix`

```python
for wrong, correct in corrections.items():
    df["report_text"] = df["report_text"].str.replace(
        wrong,
        correct,
        regex=False
    )
```

---

### Verify:

After every cleaning step, view the top five rows.

```python
df.head()
```

Never clean blindly. Always verify.

---

### Build a aleaning `module`

Create

```text
src/

preprocessing.py
```

Explanation: Should `DataLoader` also clean data? `No`.

Again `One Responsibility`.

---

Example

```python
import pandas as pd


class DataCleaner:

    def remove_duplicates(self, df):
        return df.drop_duplicates()

    def remove_missing(self, df):
        return df.dropna()

    def strip_spaces(self, df):
        df["report_text"] = df["report_text"].str.strip()
        return df

    def lowercase(self, df):
        df["report_text"] = df["report_text"].str.lower()
        return df
```

Notice that each method does `ONE`thing. Professional software.

---

### Update `main.py`

```python
from src.data_loader import DataLoader
from src.preprocessing import DataCleaner

loader = DataLoader()
cleaner = DataCleaner()

df = loader.load_data()

print("Before Cleaning")
print(df.shape)

df = cleaner.remove_missing(df)
df = cleaner.remove_duplicates(df)
df = cleaner.strip_spaces(df)
df = cleaner.lowercase(df)

print("After Cleaning")
print(df.shape)
```

---

### Final verification

Run:

```python
print(df.isnull().sum())
print(df.duplicated().sum())
print(df.head())
```

Everything should now be consistent.

---

### Engineering lesson

Always note that **cleaning is not about applying every possible transformation**. It's about making informed decisions based on the data and the business context. For example:

* Removing duplicates is usually safe when they are accidental.
* Filling missing values may be appropriate for metadata (`reported_by`) but not for the core text (`report_text`) if that text is the feature used for training.
* Converting text to lowercase is helpful for many NLP pipelines because it reduces unnecessary variation.
* Correcting spelling mistakes improves consistency but should be done carefully to avoid changing legitimate technical terms.

Every transformation should answer two questions:

1. **Why is this change necessary?**
2. **How will it improve the quality of the data for the model?**

If you learn to justify every preprocessing step rather than memorizing functions like `dropna()` or `fillna()`, you'll develop the analytical mindset expected of professional AI and Machine Learning engineers. This session naturally prepares you for **Phase 3**, where you'll convert this clean, reliable dataset into features that a machine learning model can understand.